In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu

In [ ]:
!git clone https://github.com/yuvnahr/Persistent_Mem_PRIMA.git
%cd Persistent_Mem_PRIMA

In [ ]:
!python scripts/init_db.py
!python scripts/reset_db.py
!python scripts/seed_dummy_memory.py

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3.5-35B-A3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True
)

print("Model loaded")

In [ ]:
from src.prima_memory.core.memory_store import MemoryStore
from src.prima_memory.core.embedding import EmbeddingIndex
from src.prima_memory.core.retriever import MemoryRetriever
from src.prima_memory.core.linker import MemoryLinker
from src.prima_memory.core.evolution import MemoryEvolver
from src.prima_memory.core.note import MemoryNote

store = MemoryStore()
embedder = EmbeddingIndex()

retriever = MemoryRetriever(store=store, embedder=embedder)
linker = MemoryLinker(store=store)
evolver = MemoryEvolver(store=store)

print("PRIMA ready")

In [ ]:
BASELINE_SYSTEM_PROMPT = """
You are a precise and honest assistant.

Rules:
- Only answer using known information.
- If unsure, say "I don't know".
- Do NOT hallucinate.
- Keep answers concise and factual.
"""

PRIMA_SYSTEM_PROMPT = """
You are a memory-augmented assistant.

Rules:
- Use ONLY the provided memory context when relevant.
- If memory is insufficient, say "I don't know".
- Do NOT fabricate personal details.
- Prioritize consistency with past memory.
- Be precise and grounded.
"""

In [ ]:
import torch

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
queries = [
    "My name is Rahul and I love football",
    "I work as a software engineer at Infosys",
    "I live in Bangalore",
    "My favorite food is biryani",
    "I have a dog named Bruno",

    "What is my name?",
    "Where do I work?",
    "What is my favorite food?",
    "Do I have any pets?",
    "Where do I live?",

    "I recently started learning Python",
    "I am preparing for GATE exam",
    "I want to become an AI engineer",
    "I like watching cricket",
    "I wake up at 6 AM daily",

    "What am I studying currently?",
    "What is my career goal?",
    "What time do I wake up?",
    "What do I like watching?",
    "What exam am I preparing for?",

    "I visited Goa last summer",
    "My best friend is Arjun",
    "I use a MacBook Pro",
    "I prefer tea over coffee",
    "I go to gym 4 times a week",

    "Where did I travel recently?",
    "Who is my best friend?",
    "What laptop do I use?",
    "Do I prefer tea or coffee?",
    "How often do I go to gym?",

    "I am allergic to peanuts",
    "I enjoy coding at night",
    "I have two siblings",
    "My favorite subject is math",
    "I play chess on weekends",

    "What am I allergic to?",
    "How many siblings do I have?",
    "What is my favorite subject?",
    "When do I code?",
    "What do I do on weekends?",

    "I want to improve my communication skills",
    "I struggle with time management",
    "I like traveling solo",
    "I am interested in startups",
    "I use Linux occasionally",

    "What skill do I want to improve?",
    "What do I struggle with?",
    "Do I like traveling alone?",
    "What am I interested in?",
    "Which OS do I use sometimes?"
]

In [ ]:
baseline_results = []

for q in queries:
    prompt = BASELINE_SYSTEM_PROMPT + "\nUser: " + q
    response = generate(prompt)

    baseline_results.append({
        "query": q,
        "response": response
    })

In [ ]:
import datetime

prima_results = []

for q in queries:

    memories = retriever.retrieve(q, top_k=5, expand_links=True)

    context = "\n".join([m.content for m in memories])

    prompt = PRIMA_SYSTEM_PROMPT + f"\nMemory:\n{context}\nUser: {q}"

    response = generate(prompt)

    embedding = embedder.embed_text(q)

    note = MemoryNote(
        content=q,
        context="user interaction",
        tags=["user"],
        keywords=["memory"],
        created_at=datetime.datetime.utcnow().isoformat()
    )
    note.embedding = embedding

    store.insert_memory(
        memory_id=note.id,
        content=note.content,
        created_at=note.created_at,
        context=note.context,
        keywords=note.keywords,
        tags=note.tags,
        embedding=note.embedding
    )

    embedder.add(note.id, note.embedding)

    linker.link(note, [(m, 1.0) for m in memories])
    evolver.evolve(note, memories)

    prima_results.append({
        "query": q,
        "response": response,
        "memory_used": len(memories)
    })

In [ ]:
def evaluate(baseline, prima):

    results = []

    for b, p in zip(baseline, prima):

        correctness = int(len(p["response"]) > 0)
        memory_usage = p["memory_used"]

        groundedness = int(memory_usage > 0)

        results.append({
            "query": b["query"],
            "baseline_len": len(b["response"]),
            "prima_len": len(p["response"]),
            "memory_used": memory_usage,
            "grounded": groundedness,
            "correct": correctness
        })

    return results

metrics = evaluate(baseline_results, prima_results)
metrics[:5]

In [ ]:
import json

with open("experiments/prima_vs_baseline/baseline_outputs.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

with open("experiments/prima_vs_baseline/prima_outputs.json", "w") as f:
    json.dump(prima_results, f, indent=2)

with open("experiments/prima_vs_baseline/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved all results")